# 05 — Stability and monitoring

How long until this model needs retraining? The answer here is not the one the textbook ordering suggests.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from src.config import load_config

cfg = load_config()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

import pickle
from src.splits import assign_split
from src.data_load import months_of_credit_history
from src.monitoring import performance_by_period, score_drift, retraining_signal
from src.metrics import psi

def prep(d):
    d = d.copy()
    d['credit_history_months'] = months_of_credit_history(d['earliest_cr_line'], d['issue_dt'])
    return d

with open('../data/processed/fitted.pkl','rb') as f:
    fit = pickle.load(f)
sc, NUM = fit['scorecard'], fit['NUM']
samp = prep(pd.read_parquet('../data/processed/modelling_sample.parquet'))
samp['split'] = assign_split(samp, cfg)
drift = prep(pd.read_parquet('../data/processed/drift_window.parquet'))
samp['score'] = sc.predict_proba(samp); drift['score'] = sc.predict_proba(drift)
baseline = samp.loc[samp['split']=='train','score']

## 1. Score drift — works without labels

This is what makes the 2016–2018 window useful despite incomplete labels: measuring distribution shift needs no outcome at all.

In [ ]:
allf = pd.concat([samp[['issue_dt','score','target']], drift[['issue_dt','score','target']]])
score_drift(baseline, allf, 'score', freq='Y')

## 2. Where the 2018 drift sits — and what it is

PSI crosses into *watch* in 2018 (0.1006). Feature-level PSI locates it, and a **fill-rate check settles what it is**: coverage is effectively unchanged, so this is not the availability artefact the audit was built to catch.

It is real population change. The 2018 book carries higher FICO (share above 740 doubles, 7.8% → 15.9%), roughly half the bankcard utilisation, and half the recent inquiries. **LendingClub moved upmarket** — a credit-policy decision by the lender, and exactly the kind of change that invalidates a scorecard fitted on the older population.

In [ ]:
tr = samp[samp['split']=='train']; d18 = drift[drift['issue_dt'].dt.year==2018]
pd.DataFrame([{
    'feature': f,
    'psi_2018': round(psi(tr[f].to_numpy(), d18[f].to_numpy()), 4),
    'fill_train': round(tr[f].notna().mean(), 3),
    'fill_2018': round(d18[f].notna().mean(), 3),
    'median_train': round(float(tr[f].median()), 1),
    'median_2018': round(float(d18[f].median()), 1),
} for f in NUM]).sort_values('psi_2018', ascending=False)

## 3. Performance decay — only where labels are unbiased

2016–2018 apparent bad rates **must not be plotted on the same axis** as 2012–2015. Only 12% of 2018 loans have resolved, and what resolves first inside an incomplete window is disproportionately charge-offs: a default resolves in ~5–8 months, a full payoff takes the whole 36-month term. 2016 reads 19.85% at 72% resolved; 2018 reads 13.25% at 12% resolved. Neither is the true rate.

Caveat on the table below: 2012–2014 are the training cohorts, so only 2015 is held out. The honest claim is 'no decay detected', not 'proven stable'.

In [ ]:
dec = performance_by_period(samp, 'score', freq='Y')
print(dec.to_string(index=False))
print()
drift.groupby(drift['issue_dt'].dt.year).agg(
    loans=('target','size'),
    resolved_share=('target', lambda s: s.notna().mean()),
    apparent_bad_rate=('target','mean')).round(4)

## 4. The conclusion PSI cannot reach

On the test split, score PSI was **0.0023** — as stable as a number gets. Calibration over that same split was off by **1.5–2.4 percentage points**, under-predicting risk in nine of ten deciles.

A PSI-only setup reports 'all clear' exactly where the model has stopped telling the truth about absolute risk. PSI detects *input* shift; it is structurally blind to the target moving underneath a stable input distribution.

**Two recommendations follow:**

1. Monitor observed-versus-predicted bad rate by score band, *alongside* PSI rather than instead of it.
2. Express the approval policy as a **rate**, not a probability cut-off. A score-quantile policy holds approved volume fixed by construction and is immune to calibration drift. A probability threshold silently changes approved volume the moment calibration moves.

In [ ]:
retraining_signal(dec, score_drift(baseline, samp, 'score', freq='Y'))